# Training / Validation / Testing splitting

In [ ]:
# general
from scipy.stats import wasserstein_distance
from sklearn.decomposition import PCA
from pathlib import Path
import pandas as pd
import numpy as np

from src.utils import load_config
cfg = load_config()

In [ ]:
# metadata
metadata = pd.read_csv(Path(cfg["dir"]["pipeline"]) / "lstm" / "data" / "attributes" / "basin_attributes.csv", index_col=0)

metadata_target = metadata.loc[~metadata.index.str.match(r'^[PX]\d+')]  # Start with X or P followed by digits
metadata  = metadata.loc[metadata.index.str.match(r'^[PX]\d+')]   # Everything else (true)

In [ ]:
# remove training basins with less than 50% of streamflow data
data = pd.read_csv(cfg["dir"]["andeangc"] + "/AndeanGC_data_1950_2024_qc.csv", index_col=0, parse_dates=True)
data = data.loc["2000-01-01":cfg["time"]["ref_end"]]  #TODO check 1994-2000

data_available = data.notnull().sum() / data.shape[0]
data = data.loc[:, data_available > cfg["hydro"]["min_data"]] 
metadata = metadata.loc[data.columns]

# remove training basins with: cropland cover > 20%, urban cover > 5% and big dams (> 100 hm3)
metadata = metadata[metadata.cropland_cover < 20]
metadata = metadata[metadata.urban_cover < 5]
metadata = metadata[metadata.dams_count == 0]

In [ ]:
# visual inspection of the training basins
to_remove = ["X00001870", "X00002218", "X00002239", "X03802001", 
             "X03022001", "X01020003", "X03814003", "X03820003",
             "X08376001", "X00002282", "X11536001"] 

metadata = metadata[~metadata.index.isin(to_remove)]
print("Total number of basins:", len(metadata))

In [ ]:
# attributes selection: the same list the model receives as static_attributes in
# modelling/hydrological_modelling_LSTM.yml, so the split balances what the LSTM actually sees.

static_attributes = [
    "basin_cenlat", "basin_cenlon",                              # location
    "basin_area", "slope_mean", 'elev_std',                      # topography
    "shrubs_veg_cover", "forest_cover", "herbaceous_veg_cover",  # land cover
    "glacier_area_RGI60",                                        # glacier
    'prcp_mean_ERA5', 'prcp_pci_ERA5', 'ev_mean_ERA5',           # climate
    'solid_prcp_ERA5', 'frac_snow_ERA5']                         # climate

metadata_raw = metadata.copy()          # kept unstandardized for the region strata below

# rank transform instead of z-scores, applied once the basin pool is final: basin_area and glacier_area_RGI60 are strongly skewed
metadata = metadata[static_attributes].rank(pct=True)

# The distance is computed on principal components rather than on the raw columns
pca = PCA(n_components=0.95, svd_solver="full").fit(metadata)
scores = pd.DataFrame(pca.transform(metadata), index=metadata.index,
                      columns=[f"PC{i+1}" for i in range(pca.n_components_)])

print(f"{len(static_attributes)} attributes -> {pca.n_components_} components "
      f"({pca.explained_variance_ratio_.sum():.0%} of variance)")

In [ ]:
# Optimize train/validation/test split using Wasserstein distance, stratified by region.

MIN_REGION_BASINS = 10       # a region needs at least this many basins to appear in val/test
FRAC_VAL = FRAC_TEST = 0.10
N_CANDIDATES = 5000
OBJECTIVE = "minimize" # validation and test resemble the training set and the target domain

def region_of(lat):
    for name, (north, south) in cfg["regions"].items():
        if south <= lat <= north:
            return name
    return "outside"


regions = metadata_raw["basin_cenlat"].map(region_of).rename("region")
counts = regions.value_counts()

# Regions too small to hold anything out stay entirely in training
eligible = [r for r in cfg["regions"] if counts.get(r, 0) >= MIN_REGION_BASINS]

print("region strata:")
for r in cfg["regions"]:
    n = counts.get(r, 0)
    print(f"  {r:16s} {n:3d} basins  ->  {'val/test eligible' if r in eligible else 'train only'}")

quota = {r: (max(1, round(counts[r] * FRAC_VAL)), max(1, round(counts[r] * FRAC_TEST)))
         for r in eligible}
region_ids = {r: regions.index[regions == r].to_numpy() for r in eligible}


def wdist(a, b):
    """Mean Wasserstein distance across principal components."""
    return float(np.mean([wasserstein_distance(a[c], b[c]) for c in scores.columns]))


better = (lambda s, b: s < b) if OBJECTIVE == "minimize" else (lambda s, b: s > b)
best_score = np.inf if OBJECTIVE == "minimize" else -np.inf
best_split = best_seed = None

for seed in range(N_CANDIDATES):
    rng = np.random.default_rng(seed)

    # draw the val/test quota inside each eligible region, so every candidate split keeps the
    val_ids, test_ids = [], []
    for r in eligible:
        perm = rng.permutation(region_ids[r])
        n_val, n_test = quota[r]
        val_ids += list(perm[:n_val])
        test_ids += list(perm[n_val:n_val + n_test])

    held = set(val_ids) | set(test_ids)
    train_ids = [b for b in metadata.index if b not in held]

    # the full training set enters the distance
    score = (wdist(scores.loc[train_ids], scores.loc[val_ids])
             + wdist(scores.loc[val_ids], scores.loc[test_ids])
             + wdist(scores.loc[test_ids], scores.loc[train_ids]))

    if better(score, best_score):
        best_score, best_split, best_seed = score, (train_ids, val_ids, test_ids), seed

train_basins, val_basins, test_basins = best_split

print(f"\nobjective: {OBJECTIVE} over {N_CANDIDATES} candidates | best seed: {best_seed}")
print(f"Best Wasserstein distance: {best_score:.4f}")
print(f"Split sizes - Train: {len(train_basins)}, Val: {len(val_basins)}, Test: {len(test_basins)}")

composition = pd.DataFrame({
    "train":      regions.loc[train_basins].value_counts(),
    "validation": regions.loc[val_basins].value_counts(),
    "test":       regions.loc[test_basins].value_counts(),
}).reindex(list(cfg["regions"])).fillna(0).astype(int)
print("\nregional composition:")
print(composition.to_string())

In [ ]:
# create basin lists 
with open(Path("modelling") / "basins_training.txt", "w") as file: # train 
    for x in train_basins:
        file.write(x + "\n")

with open(Path("modelling") / "basins_validation.txt", "w") as file: # validation
    for x in val_basins:
        file.write(x + "\n")

with open(Path("modelling") / "basins_test.txt", "w") as file: # testing + target basins
    for x in test_basins:
        file.write(x + "\n")

with open(Path("modelling") / "basins_test_target.txt", "w") as file: # testing + target basins
    for x in metadata_target.index.tolist():
        file.write(x + "\n")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

train_sel = metadata.loc[train_basins]
val_sel   = metadata.loc[val_basins]
test_sel  = metadata.loc[test_basins]

n_cols = 4
n_rows = int(np.ceil(len(static_attributes) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, attr in enumerate(static_attributes):
    ax = axes[i]
    # Common bins across Train/Val/Test
    combined = pd.concat([train_sel[attr], val_sel[attr], test_sel[attr]])
    bins = np.histogram_bin_edges(combined.dropna(), bins=30)

    ax.hist(train_sel[attr].dropna(), bins=bins, alpha=0.55, label='Train',
            color='tab:blue', edgecolor='black', density=True)
    ax.hist(val_sel[attr].dropna(), bins=bins, alpha=0.55, label='Val',
            color='tab:green', edgecolor='black', density=True)
    ax.hist(test_sel[attr].dropna(), bins=bins, alpha=0.55, label='Test',
            color='tab:orange', edgecolor='black', density=True)

    ax.set_title(attr)
    ax.set_xlabel('Percentile rank')
    ax.set_ylabel('Percentage')
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    ax.legend()

# Hide any unused subplots
for i in range(len(static_attributes), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()